In [15]:
# Notebook 3 — Embeddings prep, splits, and feature fusion
# This notebook:
# 1) Loads cleaned data (from Notebook 1)
# 2) Re-creates pattern transactions in the same way as Notebook 2 (for consistency)
# 3) Builds a binary sparse matrix from pattern items
# 4) Computes multilingual sentence embeddings for descriptions
# 5) Fuses pattern features + embeddings
# 6) Creates stratified train/val/test splits and saves artifacts

import os
import re
import json
import math
import random
import numpy as np
import pandas as pd
from datetime import datetime
from collections import Counter

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# IO paths
DATA_IN = "data/processed/descriptions_clean.csv"
RESULTS_DIR = "results"
PROCESSED_DIR = "data/processed"
REPORTS_DIR = "docs/reports"
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(REPORTS_DIR, exist_ok=True)

# Mining configuration (keep consistent with Notebook 2)
MIN_SUP = 0.05
MAX_ITEMSET_LEN = 3
TOP_K_ITEMS_FALLBACK = 200

# Embedding configuration
# Use multilingual model to support English/Urdu/Roman Urdu
EMBED_MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
EMBED_BATCH_SIZE = 64

TIMESTAMP = datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S UTC")
print("Config ready. Paths OK. Seed set.")

Config ready. Paths OK. Seed set.


C:\Users\stran\AppData\Local\Temp\ipykernel_11144\184963049.py:43: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  TIMESTAMP = datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S UTC")


In [16]:
if not os.path.exists(DATA_IN):
    raise FileNotFoundError(f"{DATA_IN} not found. Run Notebook 1 first.")

df = pd.read_csv(DATA_IN)
print("Loaded:", DATA_IN, "| rows:", len(df))

# Expect key columns from Notebook 1
expected_cols = [
    "id","description_final","language","osm_tag_key","osm_tag_value","city","province","country"
]
missing_cols = [c for c in expected_cols if c not in df.columns]
if missing_cols:
    print("Warning: missing expected columns:", missing_cols)

# Target label as "key=value"
df["osm_label"] = df["osm_tag_key"].fillna("") + "=" + df["osm_tag_value"].fillna("")
print("Unique osm_label:", df["osm_label"].nunique())
df.head()

Loaded: data/processed/descriptions_clean.csv | rows: 3
Unique osm_label: 3


,id,osm_id,osm_geom_type,osm_tag_key,osm_tag_value,name,name_ur,alt_name,description_raw,description_final,...,wikipedia_title,wikipedia_url,lat,lon,city,province,country,language,dedup_group_id,osm_label
0,toy-1,n1,node,amenity,park,Kids Park F-7,بچوں کا پارک ایف-7,NaN,"A quiet park near the F-7 markaz, perfect for ...","A quiet park near the F-7 markaz, perfect for ...",...,NaN,NaN,33.723,73.055,Islamabad,ICT,Pakistan,Roman Urdu,0dc2b872-da697349,amenity=park
1,toy-2,w2,way,amenity,place_of_worship,Masjid-e-Quba,مسجد قبا,Quba Mosque,پرانے بازار کے قریب ایک خوبصورت مسجد۔,پرانے بازار کے قریب ایک خوبصورت مسجد۔,...,NaN,NaN,33.706,73.039,Islamabad,ICT,Pakistan,Urdu,05f94746-e0e4091e,amenity=place_of_worship
2,toy-3,n3,node,shop,mall,Centaurus,سینٹورس,The Centaurus Mall,Famous mall with food court and cinema.,Famous mall with food court and cinema.,...,The Centaurus,NaN,33.710,73.058,Islamabad,ICT,Pakistan,English,b6ece0a2-82d7bdcf,shop=mall


In [17]:
# We reproduce key helpers from Notebook 02_2_patternmining_cleaneddata_from_ASA.ipynb so this notebook is self-contained.
import nltk

# Ensure NLTK resources are available (safe to re-run)
try:
    nltk.data.find("tokenizers/punkt")
except LookupError:
    nltk.download("punkt")
try:
    nltk.data.find("taggers/averaged_perceptron_tagger")
except LookupError:
    nltk.download("averaged_perceptron_tagger")

from nltk import word_tokenize, pos_tag

# Domain lexicons and cues (Pakistan/OSM-oriented)
OSM_VALUE_SYNONYMS = {
    "amenity=mosque": {"mosque", "masjid", "jamia", "imam-bargah", "imambargah"},
    "amenity=school": {"school", "madrasa", "college"},
    "amenity=hospital": {"hospital", "clinic"},
    "amenity=park": {"park", "family-park", "kid-park"},
    "shop=mall": {"mall", "markaz", "center", "centaurus"},
    "amenity=bazaar": {"bazaar", "bazar", "mandi", "market"},
    "highway=chowk": {"chowk", "roundabout"},
    "amenity=restaurant": {"restaurant", "hotel", "dhaba", "eatery"},
    "amenity=cafe": {"cafe", "coffee"},
    "amenity=bank": {"bank", "atm"},
    "amenity=university": {"university", "uni", "campus"},
}

ADJECTIVE_CUES = {
    "quiet","peaceful","beautiful","old","new","historic","famous","popular","big","small",
    "family","kids","cheap","expensive","crowded","busy","clean","green",
    "khubsurat","purana","naya","bari","choti","mashhoor"
}
NOUN_CUES = {
    "park","mosque","masjid","bazaar","bazar","market","mandi","chowk","roundabout","mall","markaz",
    "school","college","university","hospital","clinic","restaurant","cafe","bank",
    "lake","trail","museum","cinema","zoo"
}
URDU_NOUN_CUES = {
    "مسجد","بازار","چوک","پارک","اسکول","کالج","ہسپتال","کلیہ","ریسٹورنٹ","بینک","چڑیاگھر","سینما"
}

def clean_text(s: str) -> str:
    if not isinstance(s, str):
        return ""
    s = s.lower()
    s = re.sub(r"\s+", " ", s).strip()
    return s

def tokenize(text: str, lang: str) -> list:
    text = clean_text(text)
    if not text:
        return []
    if lang in ("English", "Roman Urdu", "Mixed"):
        try:
            return word_tokenize(text)
        except Exception:
            return text.split()
    return text.split()

def pos_tag_safe(tokens: list, lang: str) -> list:
    if not tokens:
        return []
    if lang in ("English", "Roman Urdu", "Mixed"):
        try:
            return pos_tag(tokens)
        except Exception:
            return [(t, "X") for t in tokens]
    return [(t, "X") for t in tokens]

def extract_patterns(text: str, lang: str) -> set:
    # Returns a set of pattern items found in text
    patterns = set()
    toks = tokenize(text, lang)
    if not toks:
        return patterns

    if lang in ("English", "Mixed"):
        tagged = pos_tag_safe(toks, lang)
        # adj-noun
        for i in range(len(tagged) - 1):
            w1, p1 = tagged[i]
            w2, p2 = tagged[i+1]
            if p1.startswith("JJ") and p2.startswith("NN"):
                patterns.add(f"{w1}_{w2}")
        # noun-noun
        for i in range(len(tagged) - 1):
            w1, p1 = tagged[i]
            w2, p2 = tagged[i+1]
            if p1.startswith("NN") and p2.startswith("NN"):
                patterns.add(f"{w1}_{w2}")
        # cue singles
        for w, _ in tagged:
            if w in ADJECTIVE_CUES:
                patterns.add(f"adj:{w}")
            if w in NOUN_CUES:
                patterns.add(f"noun:{w}")

    elif lang == "Roman Urdu":
        # heuristic bigrams and singles
        for i in range(len(toks) - 1):
            w1, w2 = toks[i], toks[i+1]
            if w1 in ADJECTIVE_CUES and w2 in NOUN_CUES:
                patterns.add(f"{w1}_{w2}")
        for w in toks:
            if w in ADJECTIVE_CUES:
                patterns.add(f"adj:{w}")
            if w in NOUN_CUES:
                patterns.add(f"noun:{w}")

    elif lang == "Urdu":
        # minimal Urdu noun cues and previous token
        for i, w in enumerate(toks):
            if w in URDU_NOUN_CUES:
                patterns.add(f"noun_ur:{w}")
                if i > 0 and re.match(r"^\w+$", toks[i-1]):
                    patterns.add(f"ur_prev_{toks[i-1]}_{w}")

    else:
        for w in toks:
            if w in NOUN_CUES:
                patterns.add(f"noun:{w}")
    return patterns

In [19]:
transactions = []
labels = []
row_ids = []

for idx, row in df.iterrows():
    text = row.get("description_final", "") or ""
    lang = row.get("language", "Unknown")
    city = clean_text(row.get("city", "") or "")
    label = row.get("osm_label", "")
    rid = row.get("id", idx)

    items = set()
    # Extract patterns
    pats = extract_patterns(text, lang)
    items.update(pats)

    # City token (helps locality signals)
    if city:
        items.add(f"city:{city}")

    # Guided OSM synonyms
    text_clean = clean_text(text)
    for canonical, syns in OSM_VALUE_SYNONYMS.items():
        if any(s in text_clean for s in syns):
            items.add(f"osm_hint:{canonical}")

    transactions.append(items)
    labels.append(label)
    row_ids.append(rid)

print("Transactions:", len(transactions))
print("Non-empty transactions:", sum(1 for t in transactions if len(t) > 0))

Transactions: 3
Non-empty transactions: 3


In [20]:
# Pruning strategy:
# - Keep all 'osm_hint:' items
# - Keep items with frequency >= ceil(MIN_SUP * N)
# - If still too many, keep top-K by frequency (hints always kept)

from scipy.sparse import csr_matrix

N = len(transactions)
freq = Counter()
for t in transactions:
    freq.update(t)

freq_min = max(2, math.ceil(MIN_SUP * N))
keep_items = set([it for it, c in freq.items() if c >= freq_min or it.startswith("osm_hint:")])

if len(keep_items) > TOP_K_ITEMS_FALLBACK:
    # Separate hints (always keep)
    hints = [it for it in keep_items if it.startswith("osm_hint:")]
    others = [(it, freq[it]) for it in keep_items if not it.startswith("osm_hint:")]
    others_sorted = sorted(others, key=lambda x: x[1], reverse=True)
    space = max(0, TOP_K_ITEMS_FALLBACK - len(hints))
    top_others = [it for it, _ in others_sorted[:space]]
    keep_items = set(hints + top_others)

# Build index for features
vocab_items = sorted(list(keep_items))
item_to_idx = {it: i for i, it in enumerate(vocab_items)}

# Create CSR binary matrix for pattern items: shape [N, V]
rows, cols, data = [], [], []
for i, t in enumerate(transactions):
    for it in t:
        j = item_to_idx.get(it)
        if j is not None:
            rows.append(i)
            cols.append(j)
            data.append(1)

X_patterns = csr_matrix((data, (rows, cols)), shape=(N, len(vocab_items)), dtype=np.float32)

print("Pattern matrix shape:", X_patterns.shape)
print("Avg non-zeros per row:", (X_patterns.nnz / max(1, N)))

Pattern matrix shape: (3, 3)
Avg non-zeros per row: 2.0


In [21]:
from sklearn.model_selection import StratifiedShuffleSplit
import numpy as np
import json
import os

PROCESSED_DIR = "data/processed"

y = df["label_id"].to_numpy()  # whatever your label id vector is

# Inspect class counts
unique, counts = np.unique(y, return_counts=True)
class_counts = dict(zip(unique, counts))
print("Overall class counts:", class_counts)

# Enforce at least 1 sample per class in each split when possible.
# If some classes have too few samples, we’ll adjust split sizes or drop ultra-rare classes with a warning.

min_total_per_class = 3  # needs at least 3 total to get 1 in train/val/test
rare = [c for c, n in class_counts.items() if n < min_total_per_class]

if rare:
    print(f"Warning: classes too small for 3-way stratification: {rare}.")
    print("Strategy: keep all classes, but ensure at least 2-way presence (train+test). Val may miss a rare class.")

# First split: train+temp vs test
sss1 = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_val_idx, test_idx = next(sss1.split(np.zeros_like(y), y))
y_train_val = y[train_val_idx]

# Second split: train vs val (from train_val)
# If some classes are still too small, we’ll reduce val size.
val_size = 0.2  # of train_val
val_attempts = [0.2, 0.15, 0.1, 0.05]
for vs in val_attempts:
    try:
        sss2 = StratifiedShuffleSplit(n_splits=1, test_size=vs, random_state=42)
        train_idx_sub, val_idx_sub = next(sss2.split(np.zeros_like(y_train_val), y_train_val))
        train_idx = train_val_idx[train_idx_sub]
        val_idx = train_val_idx[val_idx_sub]
        # Check each split has >=1 class
        if len(np.unique(y[train_idx])) >= 2 and len(np.unique(y[val_idx])) >= 2 and len(np.unique(y[test_idx])) >= 2:
            print(f"Using val_size={vs} from train_val, all splits have ≥2 classes.")
            break
    except ValueError as e:
        print(f"val_size={vs} failed: {e}")
else:
    # Fallback: accept that val may have 1 class; warn
    sss2 = StratifiedShuffleSplit(n_splits=1, test_size=0.1, random_state=42)
    train_idx_sub, val_idx_sub = next(sss2.split(np.zeros_like(y_train_val), y_train_val))
    train_idx = train_val_idx[train_idx_sub]
    val_idx = train_val_idx[val_idx_sub]
    print("Fallback used: validation may have 1 class. Consider adding data or adjusting class set.")

def counts(name, idx):
    u, c = np.unique(y[idx], return_counts=True)
    print(f"{name} counts:", dict(zip(u, c)))

counts("Train", train_idx)
counts("Val", val_idx)
counts("Test", test_idx)

# Save indices for Notebook 4/5/6
split_indices = {
    "train_indices": train_idx.tolist(),
    "val_indices": val_idx.tolist(),
    "test_indices": test_idx.tolist()
}
with open(os.path.join(PROCESSED_DIR, "train_val_test_indices.json"), "w") as f:
    json.dump(split_indices, f, indent=2)

# Also save y_* arrays aligned with these indices
np.save(os.path.join(PROCESSED_DIR, "y_train.npy"), y[train_idx])
np.save(os.path.join(PROCESSED_DIR, "y_val.npy"), y[val_idx])
np.save(os.path.join(PROCESSED_DIR, "y_test.npy"), y[test_idx])

print("Stratified splits regenerated and saved.")

KeyError: 'label_id'

In [7]:
# Compute sentence embeddings using sentence-transformers (multilingual)
# We use description_final as the primary text field.
try:
    from sentence_transformers import SentenceTransformer
    import torch
except Exception as e:
    raise RuntimeError(
        "sentence-transformers is required. Install with: pip install sentence-transformers\n"
        f"Import error: {e}"
    )

device = "cuda" if torch.cuda.is_available() else "cpu"
model = SentenceTransformer(EMBED_MODEL_NAME, device=device)
print("Embedding model loaded:", EMBED_MODEL_NAME, "| device:", device)

texts = df_valid["description_final"].fillna("").astype(str).tolist()

# Batch encode for efficiency
emb_list = []
for start in range(0, len(texts), EMBED_BATCH_SIZE):
    batch = texts[start:start+EMBED_BATCH_SIZE]
    embs = model.encode(batch, show_progress_bar=False, normalize_embeddings=True)
    emb_list.append(embs)
X_embed = np.vstack(emb_list).astype(np.float32)

print("Embeddings shape:", X_embed.shape)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2 | device: cpu
Embeddings shape: (3, 384)


In [8]:
# Many linear models perform better with standardized dense features.
# We'll store both the raw (normalized) embeddings and standardized variants.
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler(with_mean=True, with_std=True)
X_embed_std = scaler.fit_transform(X_embed)

print("Standardized embeddings shape:", X_embed_std.shape)

Standardized embeddings shape: (3, 384)


In [9]:
# Build fused matrix by horizontally stacking sparse patterns and dense embeddings.
# We'll store two fused variants:
# - patterns + normalized embeddings (unit-length from model)
# - patterns + standardized embeddings (zero-mean/unit-variance)

from scipy.sparse import hstack

# Convert dense to CSR for hstack compatibility
X_embed_csr = csr_matrix(X_embed)
X_embed_std_csr = csr_matrix(X_embed_std)

X_fused_norm = hstack([X_patterns, X_embed_csr], format="csr")
X_fused_std = hstack([X_patterns, X_embed_std_csr], format="csr")

print("Fused (norm) shape:", X_fused_norm.shape)
print("Fused (std)  shape:", X_fused_std.shape)

Fused (norm) shape: (3, 387)
Fused (std)  shape: (3, 387)


In [10]:
from sklearn.model_selection import train_test_split

def safe_stratified_split(X, y, test_size=0.15, val_size=0.15, random_state=SEED):
    # First, split off test; then split remaining into train/val
    # Try stratification; if it fails (e.g., too few samples in a class), fallback to non-stratified.
    try:
        X_tmp, X_test_idx, y_tmp, y_test = train_test_split(
            np.arange(len(y)), y, test_size=test_size, random_state=random_state, stratify=y
        )
        strat_ok = True
    except ValueError:
        X_tmp, X_test_idx, y_tmp, y_test = train_test_split(
            np.arange(len(y)), y, test_size=test_size, random_state=random_state, stratify=None
        )
        strat_ok = False

    # Now split train/val from tmp
    val_ratio = val_size / (1.0 - test_size)
    try:
        X_train_idx, X_val_idx, y_train, y_val = train_test_split(
            X_tmp, y_tmp, test_size=val_ratio, random_state=random_state, stratify=y_tmp if strat_ok else None
        )
    except ValueError:
        X_train_idx, X_val_idx, y_train, y_val = train_test_split(
            X_tmp, y_tmp, test_size=val_ratio, random_state=random_state, stratify=None
        )
        strat_ok = False

    return (X_train_idx, X_val_idx, X_test_idx), (y_train, y_val, y_test), strat_ok

(idx_train, idx_val, idx_test), (y_train, y_val, y_test), strat_ok = safe_stratified_split(
    X_patterns, y, test_size=0.15, val_size=0.15, random_state=SEED
)

print(f"Stratified split ok? {strat_ok}")
print("Split sizes:", len(idx_train), len(idx_val), len(idx_test))

Stratified split ok? False
Split sizes: 1 1 1


In [11]:
def slice_sparse(mat, indices):
    if hasattr(mat, "tocsr"):
        return mat[indices]
    # For dense arrays
    return mat[indices]

splits = {
    "train": {
        "idx": idx_train,
        "y": y_train
    },
    "val": {
        "idx": idx_val,
        "y": y_val
    },
    "test": {
        "idx": idx_test,
        "y": y_test
    }
}

# Build all feature variants per split
X_pat_train = slice_sparse(X_patterns, idx_train)
X_pat_val   = slice_sparse(X_patterns, idx_val)
X_pat_test  = slice_sparse(X_patterns, idx_test)

X_emb_train = X_embed[idx_train]
X_emb_val   = X_embed[idx_val]
X_emb_test  = X_embed[idx_test]

X_embstd_train = X_embed_std[idx_train]
X_embstd_val   = X_embed_std[idx_val]
X_embstd_test  = X_embed_std[idx_test]

X_fnorm_train = slice_sparse(X_fused_norm, idx_train)
X_fnorm_val   = slice_sparse(X_fused_norm, idx_val)
X_fnorm_test  = slice_sparse(X_fused_norm, idx_test)

X_fstd_train = slice_sparse(X_fused_std, idx_train)
X_fstd_val   = slice_sparse(X_fused_std, idx_val)
X_fstd_test  = slice_sparse(X_fused_std, idx_test)

print("Prepared sliced matrices for all splits.")

Prepared sliced matrices for all splits.


In [12]:
# Save sparse matrices as .npz and dense arrays as .npy
from scipy import sparse
import joblib

# Paths
PFX = "data/processed"

# Pattern vocab and feature names
feature_info = {
    "pattern_vocab": vocab_items,  # list of item strings
    "embed_model": EMBED_MODEL_NAME,
    "fused_shapes": {
        "norm": [int(X_fused_norm.shape[0]), int(X_fused_norm.shape[1])],
        "std":  [int(X_fused_std.shape[0]), int(X_fused_std.shape[1])]
    },
    "pattern_shape": [int(X_patterns.shape[0]), int(X_patterns.shape[1])],
    "embed_dim": int(X_embed.shape[1]),
    "label_to_id": label_to_id,
    "id_to_label": id_to_label,
    "timestamp": TIMESTAMP,
}

with open(os.path.join(PFX, "feature_info.json"), "w", encoding="utf-8") as f:
    json.dump(feature_info, f, ensure_ascii=False, indent=2)

# Label arrays
np.save(os.path.join(PFX, "y_train.npy"), y_train)
np.save(os.path.join(PFX, "y_val.npy"), y_val)
np.save(os.path.join(PFX, "y_test.npy"), y_test)

# Indices for traceability to df_valid rows
np.save(os.path.join(PFX, "idx_train.npy"), idx_train)
np.save(os.path.join(PFX, "idx_val.npy"), idx_val)
np.save(os.path.join(PFX, "idx_test.npy"), idx_test)

# Pattern matrices
sparse.save_npz(os.path.join(PFX, "X_patterns_train.npz"), X_pat_train)
sparse.save_npz(os.path.join(PFX, "X_patterns_val.npz"), X_pat_val)
sparse.save_npz(os.path.join(PFX, "X_patterns_test.npz"), X_pat_test)

# Embeddings (dense)
np.save(os.path.join(PFX, "X_embed_train.npy"), X_emb_train)
np.save(os.path.join(PFX, "X_embed_val.npy"), X_emb_val)
np.save(os.path.join(PFX, "X_embed_test.npy"), X_emb_test)

# Standardized embeddings (dense)
np.save(os.path.join(PFX, "X_embed_std_train.npy"), X_embstd_train)
np.save(os.path.join(PFX, "X_embed_std_val.npy"), X_embstd_val)
np.save(os.path.join(PFX, "X_embed_std_test.npy"), X_embstd_test)

# Fused sparse matrices
sparse.save_npz(os.path.join(PFX, "X_fused_norm_train.npz"), X_fnorm_train)
sparse.save_npz(os.path.join(PFX, "X_fused_norm_val.npz"), X_fnorm_val)
sparse.save_npz(os.path.join(PFX, "X_fused_norm_test.npz"), X_fnorm_test)

sparse.save_npz(os.path.join(PFX, "X_fused_std_train.npz"), X_fstd_train)
sparse.save_npz(os.path.join(PFX, "X_fused_std_val.npz"), X_fstd_val)
sparse.save_npz(os.path.join(PFX, "X_fused_std_test.npz"), X_fstd_test)

# Save the scaler for embeddings (to standardize new data consistently)
joblib.dump(scaler, os.path.join(PFX, "embed_scaler.joblib"))

# Small Markdown report
report_md_path = os.path.join(REPORTS_DIR, f"features_splits_{datetime.utcnow().strftime('%Y%m%d')}.md")
with open(report_md_path, "w", encoding="utf-8") as f:
    f.write("### Notebook 3 — Embeddings, Pattern Features, and Splits\n\n")
    f.write(f"- Timestamp: {TIMESTAMP}\n")
    f.write(f"- Rows (valid labels): {len(df_valid)}\n")
    f.write(f"- Classes: {len(label_to_id)}\n")
    f.write(f"- Pattern vocab size: {len(vocab_items)}\n")
    f.write(f"- Embedding model: {EMBED_MODEL_NAME}\n")
    f.write(f"- Embedding dim: {X_embed.shape[1]}\n")
    f.write(f"- Split sizes: train={len(idx_train)}, val={len(idx_val)}, test={len(idx_test)}\n")
print("Artifacts saved to data/processed and docs/reports.")

Artifacts saved to data/processed and docs/reports.


C:\Users\stran\AppData\Local\Temp\ipykernel_11144\177822104.py:64: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  report_md_path = os.path.join(REPORTS_DIR, f"features_splits_{datetime.utcnow().strftime('%Y%m%d')}.md")


In [14]:
print("Notebook 02_3 complete.")
print("- Patterns: data/processed/X_patterns_{train,val,test}.npz")
print("- Embeddings: data/processed/X_embed_{train,val,test}.npy")
print("- Embeddings (std): data/processed/X_embed_std_{train,val,test}.npy")
print("- Fused (norm/std): data/processed/X_fused_{norm|std}_{train,val,test}.npz")
print("- Labels: data/processed/y_{train,val,test}.npy")
print("- Feature info: data/processed/feature_info.json")
print("- Scaler: data/processed/embed_scaler.joblib")
print("- Report: docs/reports/features_splits_YYYYMMDD.md")
print("Next: Notebook 02_4 — Baseline classifiers (patterns-only vs embeddings vs fused) + metrics.")

Notebook 02_3 complete.
- Patterns: data/processed/X_patterns_{train,val,test}.npz
- Embeddings: data/processed/X_embed_{train,val,test}.npy
- Embeddings (std): data/processed/X_embed_std_{train,val,test}.npy
- Fused (norm/std): data/processed/X_fused_{norm|std}_{train,val,test}.npz
- Labels: data/processed/y_{train,val,test}.npy
- Feature info: data/processed/feature_info.json
- Scaler: data/processed/embed_scaler.joblib
- Report: docs/reports/features_splits_YYYYMMDD.md
Next: Notebook 02_4 — Baseline classifiers (patterns-only vs embeddings vs fused) + metrics.
